## Dataset Drift Collection

### Purpose

HC3 was created using older GPT-3.5-era AI responses. To test whether our detector generalizes to newer AI-generated text, we created a separate **modern external evaluation dataset**.

This notebook collects the **human-written side** of that dataset from recent Stack Exchange posts.

### Collection goals
- Use questions and answers created from **January 1, 2025 onward**
- Collect natural-language Q&A from domains similar to HC3
- Keep only questions that have at least one answer
- Select one human answer per question:
  1. accepted answer, when available
  2. otherwise the highest-scoring answer
- Save the resulting human Q&A dataset for later curation

This dataset is used **only for external evaluation**. It is not used for model training or hyperparameter tuning.


In [1]:
import requests
import pandas as pd
from datetime import datetime, timezone
from pathlib import Path

### Define the collection window

The drift dataset should represent more recent human writing than HC3, so we restrict collection to questions created from **January 1, 2025 through the current date**.


In [2]:
START_DATE = datetime(2025, 1, 1, tzinfo=timezone.utc)
END_DATE = datetime.now(timezone.utc)

FROM_DATE = int(START_DATE.timestamp())
TO_DATE = int(END_DATE.timestamp())

print("Collection window:", START_DATE.date(), "to", END_DATE.date())

Collection window: 2025-01-01 to 2026-10-29


### Select domains

HC3 contains a mix of general, scientific, medical, and financial Q&A. Rather than recreate the exact HC3 websites, we collect newer human-written questions from comparable domains.

Each Stack Exchange site is assigned to one broad domain used later for evaluation.

In [3]:
sites = {
    "science": "physics",
    "biology": "biology",
    "medicine": "medicalsciences",
    "finance": "money",
    "general": "earthscience"
}

sites

{'science': 'physics',
 'biology': 'biology',
 'medicine': 'medicalsciences',
 'finance': 'money',
 'general': 'earthscience'}

### Collect candidate questions

For each domain, we request recent questions from the Stack Exchange API.

We keep the question ID, title, full question body, score, answer count, creation date, tags, domain, and source site. At this stage we collect a larger candidate pool so weak or unusable examples can be removed later.

In [4]:
def get_questions(site, domain, pagesize=50):
    url = "https://api.stackexchange.com/2.3/questions"

    params = {
        "site": site,
        "fromdate": FROM_DATE,
        "todate": TO_DATE,
        "sort": "votes",
        "order": "desc",
        "min": 1,
        "pagesize": pagesize,
        "filter": "withbody"
    }

    response = requests.get(url, params=params)
    response.raise_for_status()

    rows = []

    for q in response.json().get("items", []):
        rows.append({
            "question_id": q["question_id"],
            "question": q["title"],
            "question_body": q.get("body"),
            "question_score": q["score"],
            "answer_count": q["answer_count"],
            "creation_date": q["creation_date"],
            "tags": q["tags"],
            "domain": domain,
            "source": site
        })

    return pd.DataFrame(rows)

In [5]:
candidate_frames = []

for domain, site in sites.items():
    domain_questions = get_questions(site, domain)
    candidate_frames.append(domain_questions)
    print(f"{domain}: {len(domain_questions)} questions")

candidates = pd.concat(candidate_frames, ignore_index=True)

print(f"\nTotal candidates: {len(candidates)}")
candidates.head()

science: 50 questions
biology: 50 questions
medicine: 50 questions
finance: 50 questions
general: 50 questions

Total candidates: 250


,question_id,question,question_body,question_score,answer_count,creation_date,tags,domain,source
0,848666,What is &quot;Induced Atmospheric Vibration&qu...,<p>The blackout seen today on the Iberian Peni...,71,8,1745854998,"[electricity, atmospheric-science, vibrations]",science,physics
1,870148,Air pressure on Noah&#39;s Ark,<p>My mother asked me today: &quot;If Noah's A...,51,2,1773634607,"[pressure, fluid-statics, atmospheric-science]",science,physics
2,853814,Why are there shadows surrounding these underw...,<p>I took this picture of a water feature on c...,43,1,1750023727,"[optics, visible-light, everyday-life, geometr...",science,physics
3,868126,Looking for an old physics lab meme (early 2000s),<p>I’m trying to track down a mock physics lab...,42,1,1768871082,"[soft-question, error-analysis, data-analysis,...",science,physics
4,854111,Are there classical physics phenomena that rem...,"<p>Most of the major <a href=""https://en.wikip...",33,12,1750370342,"[classical-mechanics, big-list]",science,physics


### Apply basic question filters

The external benchmark should contain questions that can be answered in ordinary prose.

Before fetching answers, we remove:

- questions with no answers
- very short question titles that are more likely to depend heavily on missing context

More detailed curation is performed after the raw human dataset is collected.

In [6]:
filtered_candidates = candidates[
    (candidates["answer_count"] > 0) &
    (candidates["question"].str.len() >= 25)
].copy()

print("Candidates before filtering:", len(candidates))
print("Candidates after filtering:", len(filtered_candidates))
print("\nCandidates by domain:")
print(filtered_candidates["domain"].value_counts().sort_index())

Candidates before filtering: 250
Candidates after filtering: 214

Candidates by domain:
domain
biology     45
finance     50
general     39
medicine    32
science     48
Name: count, dtype: int64


### Fetch human answers

The Stack Exchange API allows answers to be requested in batches using question IDs.

We retain the answer text along with metadata such as score, acceptance status, creation date, and question ID.

In [7]:
def get_answers_for_questions(question_ids, site):
    all_answers = []

    for i in range(0, len(question_ids), 100):
        batch = question_ids[i:i + 100]
        ids = ";".join(str(qid) for qid in batch)

        url = f"https://api.stackexchange.com/2.3/questions/{ids}/answers"

        params = {
            "site": site,
            "sort": "votes",
            "order": "desc",
            "pagesize": 100,
            "filter": "!nNPvSNdWme"
        }

        response = requests.get(url, params=params)
        response.raise_for_status()

        all_answers.extend(response.json().get("items", []))

    return pd.DataFrame(all_answers)

In [8]:
answer_frames = []

for site in filtered_candidates["source"].unique():
    site_questions = filtered_candidates[
        filtered_candidates["source"] == site
    ]

    question_ids = site_questions["question_id"].tolist()

    site_answers = get_answers_for_questions(
        question_ids,
        site
    )

    site_answers["source"] = site
    answer_frames.append(site_answers)

answers_df = pd.concat(answer_frames, ignore_index=True)

print(f"Answers collected: {len(answers_df)}")
answers_df[
    ["question_id", "answer_id", "score", "is_accepted", "creation_date"]
].head()

Answers collected: 360


,question_id,answer_id,score,is_accepted,creation_date
0,868126,868128,63,False,1768871621
1,848666,848701,59,False,1745883539
2,853814,853815,59,True,1750023727
3,870148,870151,55,True,1773641722
4,842313,842315,49,False,1739042449


### Select one human answer per question

A question may have multiple human answers. To make the dataset consistent, we use the following rule:

1. Prefer an **accepted answer**
2. If no answer was accepted, use the **highest-scoring answer**

Sorting by question ID, acceptance status, and score lets us select one answer per question without using a custom groupby function.


In [9]:
best_answers = (
    answers_df
    .sort_values(
        ["question_id", "is_accepted", "score"],
        ascending=[True, False, False]
    )
    .drop_duplicates(
        subset="question_id",
        keep="first"
    )
    .reset_index(drop=True)
)

print(f"Questions with a selected answer: {len(best_answers)}")

Questions with a selected answer: 213


### Merge questions with selected answers

We merge the question metadata with the selected human answer using `question_id`.

The result is the raw human Q&A dataset that will be cleaned and curated into the final 150-question external benchmark.


In [10]:
human_dataset = filtered_candidates.merge(
    best_answers[
        [
            "question_id",
            "body",
            "score",
            "creation_date",
            "is_accepted"
        ]
    ],
    on="question_id",
    how="inner",
    suffixes=("_question", "_answer")
)

print(f"Human Q&A pairs collected: {len(human_dataset)}")

human_dataset[
    [
        "domain",
        "question",
        "question_score",
        "score",
        "is_accepted"
    ]
].head(10)

Human Q&A pairs collected: 213


,domain,question,question_score,score,is_accepted
0,science,What is &quot;Induced Atmospheric Vibration&qu...,71,14,True
1,science,Air pressure on Noah&#39;s Ark,51,55,True
2,science,Why are there shadows surrounding these underw...,43,59,True
3,science,Looking for an old physics lab meme (early 2000s),42,63,False
4,science,Are there classical physics phenomena that rem...,33,30,False
5,science,How far does a submarine need to dive to not f...,33,30,True
6,science,How do we know what our measuring tools are me...,31,30,False
7,science,What is the physics behind how welding works?,28,30,True
8,science,How quickly does light fall in a hall of mirrors?,27,14,False
9,science,Why doesn&#39;t sugar spontaneously switch chi...,26,12,False


### Save the raw human dataset

The raw collection is saved before manual/automated curation so the original API results remain reproducible.

The curated dataset used for the final drift evaluation contains **150 human Q&A pairs**, balanced across five domains (30 per domain). Those same 150 questions are later answered independently by ChatGPT, Claude, and Gemini.


In [11]:
output_path = Path("../../data/modern_human_qa_raw.csv")

human_dataset.to_csv(
    output_path,
    index=False
)

print(f"Saved {len(human_dataset)} rows to {output_path}")

Saved 213 rows to ../data/modern_human_qa_raw.csv


### Next step: build the external drift benchmark

After this raw collection step:

1. Clean HTML and formatting from the human answers
2. Remove context-dependent, highly specialized, or otherwise unsuitable examples
3. Select **30 questions from each of the five domains** for 150 total human examples
4. Generate new answers to the same questions using:
   - ChatGPT
   - Claude
   - Gemini
5. Combine the four answer sources into the final external evaluation dataset

The completed benchmark contains:

- 150 human answers
- 150 ChatGPT answers
- 150 Claude answers
- 150 Gemini answers
- **600 total responses**

Model performance on this dataset is evaluated separately from HC3 training and validation so it measures generalization to newer data rather than training performance.
